<a href="https://colab.research.google.com/github/snehasuresh4836-stack/Fakenews_detecter_AI/blob/main/fake_news_detecter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
!pip install -q pandas numpy scikit-learn gradio nltk requests

In [86]:

import pandas as pd
import numpy as np
import re
import nltk
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download stopwords for text cleaning
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [87]:
# Step 2: Define a text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)          # keep only letters
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)


In [88]:
# Step 3: Load and balance your dataset
import pandas as pd
from sklearn.utils import resample

# Load the dataset
df = pd.read_csv("/content/news_clean_onlly.csv")

# Standardize column names
df.columns = [c.strip().lower() for c in df.columns]

# Normalize label column
df['label'] = df['label'].astype(str).str.strip().str.upper()

# Combine title + clean_text
df['combined_text'] = (df['title'].fillna('') + ' ' + df['clean_text'].fillna('')).apply(clean_text)

# Check label distribution
print("Original label counts:\n", df['label'].value_counts())

# Balance the dataset
fake_df = df[df['label'] == 'FAKE']
real_df = df[df['label'] == 'TRUE'] # Changed 'REAL' to 'TRUE' to match actual labels

if len(real_df) == 0:
    raise ValueError("No TRUE-labeled rows found. Check your label column formatting.") # Changed message

real_upsampled = resample(real_df, replace=True, n_samples=len(fake_df), random_state=42)
df_balanced = pd.concat([fake_df, real_upsampled])

print("Balanced label counts:\n", df_balanced['label'].value_counts())

Original label counts:
 label
TRUE    21170
FAKE    17346
Name: count, dtype: int64
Balanced label counts:
 label
FAKE    17346
TRUE    17346
Name: count, dtype: int64


In [89]:
from sklearn.model_selection import train_test_split

X = df['combined_text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(X_train), "Test size:", len(X_test))


Train size: 30812 Test size: 7704


In [90]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=200, n_jobs=-1))
])

model.fit(X_train, y_train)
print("Model trained successfully!")


Model trained successfully!


In [91]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9867601246105919

Classification Report:
               precision    recall  f1-score   support

        FAKE       0.99      0.98      0.99      3470
        TRUE       0.98      0.99      0.99      4234

    accuracy                           0.99      7704
   macro avg       0.99      0.99      0.99      7704
weighted avg       0.99      0.99      0.99      7704


Confusion Matrix:
 [[3395   75]
 [  27 4207]]


In [92]:
import numpy as np

def analyze_news(title, text):
    combined = clean_text((title or '') + ' ' + (text or ''))
    if not combined.strip():
        return "UNKNOWN\nConfidence: 0.0\nNo content provided."

    pred = model.predict([combined])[0]
    proba = model.predict_proba([combined])[0]
    confidence = float(np.max(proba))

    verdict = f"Verdict: {pred}\nConfidence: {confidence:.2f}"
    if pred == "FAKE":
        verdict += "\n⚠️ The news is FAKE."
    else:
        verdict += "\n✅ The news is REAL."
    return verdict

In [93]:
NEWS_API_KEY = "ccd375b4284b4348859285f22eb58535"


In [94]:
import gradio as gr
import requests

import requests

NEWS_API_KEY = "ccd375b4284b4348859285f22eb58535"  # your real key

def search_latest_news(query, language='en', page_size=5):
    endpoint = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "language": language,
        "pageSize": page_size,
        "sortBy": "publishedAt",
        "apiKey": NEWS_API_KEY
    }
    try:
        r = requests.get(endpoint, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        articles = []
        for a in data.get("articles", []):
            articles.append({
                "source": a.get("source", {}).get("name", ""),
                "title": a.get("title", ""),
                "url": a.get("url", "")
            })
        return articles
    except Exception as e:
        return [{"source": "Error", "title": "Could not fetch news.", "url": str(e)}]



In [95]:
def chatbot_interface(title, text, query):
    # Step 1: classify the pasted news
    result = analyze_news(title, text)

    # Step 2: decide what to search
    search_term = query if query else (title or '') + " " + (text or '')

    # Step 3: fetch live coverage
    articles = search_latest_news(search_term) if search_term.strip() else []
    if articles:
        result_str = "\n".join([
            f"- **{a.get('source','')}**: [{a.get('title','')}]({a.get('url','')})"
            for a in articles if a.get('url')
        ])
    else:
        result_str = "⚠️ No recent coverage found. Please verify with trusted sources."

    return f"{result}\n\n### 📰 Recent coverage for '{search_term}':\n{result_str}"


In [96]:
demo = gr.Interface(
    fn=chatbot_interface,
    inputs=[
        gr.Textbox(label="News Title"),
        gr.Textbox(label="News Text", lines=8),
        gr.Textbox(label="Search Query for Today's News (optional)")
    ],
    outputs=gr.Markdown(label="Result"),
    title="Student Fake News Detector"
)

demo.launch(debug=True)



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c0267b93e1ef98cf46.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c0267b93e1ef98cf46.gradio.live
